# **<font color='#6edce9'>ETL: Consolidación Histórica y Homologación de Dotación de RRHH (2019-2025)</font>**
### **Objetivo:** Construir el pipeline de mapeo y limpieza para unificar los datasets anuales de RRHH y migrar la base de datos consolidada a PostgreSQL.

### **Criterios Clave del Proyecto:**
* **Limpieza Estricta:** Eliminación de espacios en blanco (`.strip()`) en nombres de columnas para evitar fallas de indexación.
* **Preservación de Identificadores:** Carga de campos críticos (`RENAES`, `id_cargo`) estrictamente como texto (`str`) para no perder ceros a la izquierda.
* **Optimización de Memoria:** Uso de lecturas parciales (`nrows=0`) durante la fase de mapeo estructural.
* **Destino DB:** Preparación de variables aptas para la sintaxis nativa de PostgreSQL.


## **<font color='#eda985'> 1. Configuración y Funciones Globales </font>**

In [219]:
import pandas as pd
from sqlalchemy import create_engine

# =============================================================================
# A. CONFIGURACIÓN ESTRUCTURAL (DICCIONARIO DE ARCHIVOS)
# =============================================================================
# A.1. Diccionario de rutas y fila de header por año
archivos = {
    2019: ('../data/raw/BASE_DIC_2019.xlsx', 5),
    2020: ('../data/raw/BASE_DIC_2020.xlsx', 0),
    2021: ('../data/raw/BASE_DIC_2021.xlsx', 0),
    2022: ('../data/raw/BASE_DIC_2022.xlsx', 0),
    2023: ('../data/raw/BASE_DIC_2023.xlsx', 0),
    2024: ('../data/raw/BASE_DIC_2024.xlsx', 1),
    2025: ('../data/raw/BASE_DIC_2025.xlsx', 0)
}

In [220]:
# =============================================================================
# B. FUNCIONES GLOBALES
# =============================================================================

# B.1. FUNCIÓN PARA LECTURA RÁPIDA DE COLUMNAS (FASE DE MAPEO)

def obtener_columnas_iniciales(ruta, header_row):
    """Lee solo la cabecera del Excel para no saturar memoria RAM."""
    df_temp = pd.read_excel(ruta, header=header_row, nrows=0)
    # Limpia espacios y descarta columnas vacías/automáticas
    return [str(c).strip() for c in df_temp.columns if c and not str(c).startswith('Unnamed:')]


In [221]:
# B.2. FUNCIÓN DE COMPARACIÓN DE METADATOS (AÑO VS AÑO)

def comparar_columnas_anios(columnas_anio_a, columnas_anio_b, anio_a,  anio_b):
    """Compara estructuralmente las columnas de dos años para detectar discrepancias."""
    cols_a = set(columnas_anio_a)
    cols_b = set(columnas_anio_b)

    solo_en_a = cols_a - cols_b
    solo_en_b = cols_b - cols_a
    en_ambos = cols_a & cols_b

    print(f'---- Coinciden exactamente ({len(en_ambos)}) ----')
    for c in sorted(en_ambos): print(' ', c)
    print(f'---- Solo en {anio_a} ({len(solo_en_a)}) ----')
    for c in sorted(solo_en_a): print(' ', c)
    print(f'---- Solo en {anio_b} ({len(solo_en_b)}) ----')
    for c in sorted(solo_en_b): print(' ', c)

In [222]:
# B.3. FUNCION DE COMPARA VALORES UNICOS (CAMPO VS CAMPO)

def comparar_valores(df_a, col_a, nombre_a, df_b, col_b, nombre_b):
    """Compara el contenido real y valores únicos de dos columnas sospechosas."""
    vals_a = sorted(df_a[col_a].dropna().unique().tolist())
    vals_b = sorted(df_b[col_b].dropna().unique().tolist())

    print(f"{nombre_a} ({col_a}): {len(vals_a)} valores únicos")
    print(f"{nombre_b} ({col_b}): {len(vals_b)} valores únicos")
    print(f"¿Son el mismo conjunto? {set(vals_a) == set(vals_b)}")
    

    return pd.DataFrame({
        f'{nombre_a} ({col_a})': pd.Series(vals_a),
        f'{nombre_b} ({col_b})': pd.Series(vals_b)
    })

In [223]:
# B.4. FUNCIÓN PARA CARGA REAL Y LIMPIEZA PROFUNDA DE DATAFRAMES

def cargar_dataframe_limpio(anio, config_archivos):
    """Carga el DataFrame completo forzando IDs como texto y limpiando columnas."""
    ruta, header_row = config_archivos[anio]
    
    # Crucial para PostgreSQL: Evita que RENAES o IDs pierdan ceros a la izquierda
    dtypes_dict = {
        'REANES FINAL': str, 'RENAES': str, 'codigo_renaes': str,
        'id_cargo': str, 'id_cargo_recod': str, 'CODCARGO': str, 
        'UBIGEO': str
    }
    
    df = pd.read_excel(ruta, header=header_row, dtype=dtypes_dict)
    df.columns = df.columns.str.strip() # Limpieza de espacios en blanco
    
    # Filtrar columnas Unnamed reales
    columnas_validas = [c for c in df.columns if c and not str(c).startswith('Unnamed:')]
    return df[columnas_validas]

In [224]:
# B.5. FUNCION PARA CONVERTIR FECHA DE NACIMIENTO A TIPO DATO DATETIME

def convertir_fecha_nacimiento(df, columna_original='fecha_nacimiento', formatos=('%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y')):
    """Convierte fecha_nacimiento a datetime, manejando formatos mixtos, texto 'no especifica' y fechas seriales de Excel."""
    nueva_columna = f'{columna_original}_dt'

    serie = df[columna_original].replace(
        to_replace=r'(?i)no especifica|n/a|sin dato|no aplica', value=pd.NA, regex=True
    )

    df[nueva_columna] = pd.NaT

    # Caso especial: fechas seriales de Excel (números puros, ej. '34385')
    es_serial = serie.astype(str).str.match(r'^\d+(\.\d+)?$', na=False)
    if es_serial.any():
        df.loc[es_serial, nueva_columna] = pd.to_datetime(
            serie[es_serial].astype(float), unit='D', origin='1899-12-30', errors='coerce'
        )

    # Formatos de texto conocidos, incluyendo variante con hora
    formatos_completos = list(formatos) + ['%Y-%m-%d %H:%M:%S']
    for formato in formatos_completos:
        pendientes = serie.notna() & df[nueva_columna].isna() & ~es_serial
        df.loc[pendientes, nueva_columna] = pd.to_datetime(
            serie.loc[pendientes], format=formato, errors='coerce'
        )

    nulos_originales = serie.isna().sum()
    nulos_despues = df[nueva_columna].isna().sum()
    no_convertidos = nulos_despues - nulos_originales

    print(f"Nulos genuinos: {nulos_originales}")
    print(f"Nulos despues de convertir: {nulos_despues}")
    print(f"No se pudieron convertir (formato desconocido): {no_convertidos}")

    return df

In [225]:
# B.6. FUNCION PARA CALCULAR EDAD DESDE LA FECHA DE NACIMIENTO

def calcular_edad(fecha_nacimiento, anio_corte):    
    """Calcula la edad entera en años al 31 de diciembre del año de corte."""
    fecha_corte = pd.Timestamp(year=anio_corte, month=12, day=31)
    edad = (fecha_corte - fecha_nacimiento).dt.days// 365
    return edad

In [226]:
# B.7. FUNCION PARA DECARTAR COLUMNAS CON PREFIJOS

def descartar_por_prefijo(anio, prefijo, motivo, columnas_por_anio):
    """
    Busca en columnas_por_anio[anio] todas las columnas que empiecen con `prefijo`
    (útil para campos cuyo nombre cambia cada año, ej. EMERGENCIA, VRAEM) y las
    agrega a columnas_descartadas[anio] con el motivo indicado.
    Devuelve la lista de columnas encontradas, para que puedas revisarlas.
    """
    encontradas = [c for c in columnas_por_anio if str(c).startswith(prefijo)]

    if not encontradas:
        print(f"⚠️ No se encontró ninguna columna con prefijo '{prefijo}' en {anio}")
        return []

    if anio not in columnas_descartadas:
        columnas_descartadas[anio] = {}

    for col in encontradas:
        columnas_descartadas[anio][col] = motivo
        print(f"{anio}: descartada -> {repr(col)}")

    return encontradas

In [227]:
# B.8. FUNCION PARA IDENTIFICAR COLUMNAS INICIALES QUE SE MANTIENE - NO RENOMBRADAS NI DESCARTADAS

def columnas_que_se_mantienen(columnas_del_anio, anio):
    """Columnas que no requieren rename ni fueron descartadas: se mantienen tal cual."""
    renombradas = set(mapeo_columnas.get(anio, {}).keys())
    descartadas = set(columnas_descartadas.get(anio, {}).keys())
    return sorted(set(columnas_del_anio) - renombradas - descartadas)

In [228]:
# B.9. FUNCIÓN PARA PROCESAR Y VALIDAR COLUMNAS FINALES (POST-MAPEO)

def obtener_columnas_finales(columnas_del_anio, anio, mapeo_columnas, columnas_descartadas):
    """Devuelve los nombres definitivos tras aplicar reglas de negocio y descartes."""
    descartadas = set(columnas_descartadas.get(anio, {}).keys())
    mapeo = mapeo_columnas.get(anio, {})

    mantenidas = set(columnas_del_anio) - descartadas
    finales = {mapeo.get(col, col) for col in mantenidas}
    return sorted(finales)

In [229]:
# B.10. FUNCIÓN DE CONSOLIDACIÓN Y ESTANDARIZACIÓN (FASE ETL)

def consolidar_año(df, anio, mapeo_columnas, columnas_descartadas):
    """Aplica descartes, renombra columnas y añade la etiqueta del año correspondiente."""
    # Clonar para no alterar el DataFrame original en memoria
    df_proc = df.copy()
    
    # 1. Descartar no deseadas (filtrando solo las que existan en el DF actual)
    descartar = [c for c in columnas_descartadas.get(anio, {}).keys() if c in df_proc.columns]
    df_proc.drop(columns=descartar, inplace=True)
    
    # 2. Renombrar según el diccionario canónico
    mapeo = mapeo_columnas.get(anio, {})
    df_proc.rename(columns=mapeo, inplace=True)
    
    # 3. Formatear nombres a minúsculas y snake_case para compatibilidad PostgreSQL
    df_proc.columns = df_proc.columns.str.lower().str.replace(' ', '_').str.replace('+', 'mas')
    
    # 4. Trazabilidad: Inyectar el año del registro
    df_proc['anio_registro'] = anio
    return df_proc

In [230]:
# B.11. FUNCIÓN DE INGESTA A BASE DE DATOS (POSTGRESQL)

def exportar_a_postgresql(df_consolidado, tabla_destino, usuario, password, host, puerto, bd):
    """Realiza la carga masiva indexada del dataset unificado a PostgreSQL."""
    str_conexion = f'postgresql://{usuario}:{password}@{host}:{puerto}/{bd}'
    engine = create_engine(str_conexion)
    
    print(f"Iniciando volcado masivo en la tabla '{tabla_destino}'...")
    # chunksize fragmenta la carga para evitar colapsar la memoria de la BD
    df_consolidado.to_sql(name=tabla_destino, con=engine, if_exists='replace', index=False, chunksize=5000)
    print("¡Conexión y carga masiva finalizada con éxito en PostgreSQL!")

## **<font color='#eda985'>  2. Fase de Exploración y Mapeo Extremo: 2019 vs 2025 </font>**

In [231]:
# =============================================================================
# 2. FASE DE EXPLORACIÓN Y MAPEO EXTREMO: 2019 VS 2025
# =============================================================================

# 2.1 Extracción e inspección rápida de nombres de columnas

columnas_2019 = obtener_columnas_iniciales(archivos[2019][0], archivos[2019][1])
columnas_2025 = obtener_columnas_iniciales(archivos[2025][0], archivos[2025][1])

print("1. COMPARANDO ESTRUCTURA DE NOMBRES:\n")
comparar_columnas_anios(columnas_2019, columnas_2025, 2019, 2025)

1. COMPARANDO ESTRUCTURA DE NOMBRES:

---- Coinciden exactamente (22) ----
  CATEGORIA
  DEPARTAMENTO
  DIRESA
  DISTRITO
  ESTRATEGICOS
  MICRORRED
  PCM
  PEA
  PLIEGO
  PROVINCIA
  Quintil
  RED
  TIPO
  UBIGEO
  UE
  condicion_especialidad
  condicion_laboral
  es_especialista
  especialidad
  id_especialidad
  regimen_laboral
  sexo
---- Solo en 2019 (24) ----
  APS 2015
  CALSIFICACION
  CARGO_ESTRUCTURAL
  CODCARGO
  DESCRIPCION ESTABLECIMIENTO
  DESCRIPCION PLIEGO
  Dist Frontera
  EMERGENCIA
 (*1*)D.S. 136-2019-PCM 
Desde el: 26 Julio  Hasta el: 24 Set 
y (*2*) D.S. 135-2019-PCM 
Desde el: 28 Julio, Hasta el: 25 Set
(*3*) D.S. 137-2019-PCM 
Desde el: 27 Julio, Hasta el: 24 Set
  ESTADO
  Grupo Final
  Grupo Final 2
  INSTITUCION
  MICRORRED PRIORIZADA APS
  PLIEGO + DESCRIP
  REANES FINAL
  UE + DESCRIP UE
  UNIDAD EJECUTORA
  VRAEM 2016 (DS 040-2016-PCM)
  VRAEM 2017 (DS 112-2017-PCM)
  ZAF 2014 FINAL
  cargo
  fecha_nacimiento
  id_cargo
  id_condicion_especialidad
---- Solo

In [232]:
# 2.2 CARGA REAL DE DATAFRAMES (Obligatorio para poder analizar los valores internos)

print("\nCargando DataFrames completos para auditoría de contenido...")
df_2019 = cargar_dataframe_limpio(2019, archivos)
df_2025 = cargar_dataframe_limpio(2025, archivos)
print("\nCarga Finalizada...")


Cargando DataFrames completos para auditoría de contenido...

Carga Finalizada...


In [233]:
# Veridicando carga
df_2019.count()

PEA                                                                                                                                                                                                              216454
REANES FINAL                                                                                                                                                                                                     216454
INSTITUCION                                                                                                                                                                                                      198434
PLIEGO                                                                                                                                                                                                           216454
DESCRIPCION PLIEGO                                                                                                                      

In [234]:
# Veridicando carga
df_2025.count()

DOBLEEMPLEO                                                         286184
RENAES                                                              286184
PLIEGO                                                              286184
DESCRIPCIONPLIEGO                                                   286184
PLIEGODESCRIP                                                       286184
UE                                                                  286184
UNIDADEJECUTORA                                                     286184
UEDESCRIPUE                                                         286184
UBIGEO                                                              286184
DEPARTAMENTO                                                        286184
PROVINCIA                                                           286184
DISTRITO                                                            286184
DIRESA                                                              286184
RED                      

In [235]:
# ---- Verificando datos de fecha de nacimiento excel 2019 ----

df_2019['fecha_nacimiento'].dtype

<StringDtype(storage='python', na_value=nan)>

In [236]:
# ---- Verificando datos de fecha de nacimiento excel 2019 ----
# df_2019[['fecha_nacimiento']]
df_2019['fecha_nacimiento'].head(10)

0    28/02/1980
1    23/06/1987
2    21/12/1986
3    05/12/1953
4    22/04/1954
5    25/12/1954
6    19/05/1956
7    30/09/1975
8    05/06/1961
9    05/03/1958
Name: fecha_nacimiento, dtype: str

In [237]:
# 2.3 CORRECCIÓN DE COLUMNAS E INYECCIÓN DE EDAD 
# Limpiamos las columnas reales del DF por si acaso quedaran residuos
df_2019.columns = df_2019.columns.str.strip()
df_2025.columns = df_2025.columns.str.strip()

# Convertimos fecha y calculamos EDAD en el DF de 2019
df_2019 = convertir_fecha_nacimiento(df_2019)
df_2019['EDAD'] = calcular_edad(df_2019['fecha_nacimiento_dt'], 2019)

Nulos genuinos: 726
Nulos despues de convertir: 726
No se pudieron convertir (formato desconocido): 0


In [238]:
# ---- Verificando el resultado ---
df_2019['EDAD'].head(10)

0    39.0
1    32.0
2    33.0
3    66.0
4    65.0
5    65.0
6    63.0
7    44.0
8    58.0
9    61.0
Name: EDAD, dtype: float64

In [239]:
# ---- Verificando el resultado ---
df_2019['EDAD'].describe()
#df_2019['EDAD'].isna().sum()

count    215728.000000
mean         43.913715
std          12.059142
min          17.000000
25%          34.000000
50%          43.000000
75%          53.000000
max          86.000000
Name: EDAD, dtype: float64

In [240]:
# Verificando los cargos de los registros con edad menores de 20 años
df_2019[df_2019['EDAD'] < 20][['EDAD', 'cargo']].value_counts()

EDAD  cargo                                     
19.0  AUXILIAR ADMINISTRATIVO                       16
      DIGITADOR/A                                   13
      TRABAJADOR/A DE SERVICIOS GENERALES           10
18.0  AUXILIAR ADMINISTRATIVO                        4
      TRABAJADOR/A DE SERVICIOS GENERALES            4
19.0  TECNICO/A EN SEGURIDAD                         2
      TECNICO/A EN SERVICIOS GENERALES I             1
      TECNICO/A EN SOPORTE INFORMATICO               1
      TECNICO/A EN ENFERMERIA I                      1
18.0  TECNICO/A EN SERVICIOS GENERALES I             1
      AUXILIAR DE NUTRICION                          1
17.0  ESPECIALISTA EN SOPORTE INFORMATICO            1
19.0  TECNICO/A EN NUTRICION I                       1
      ENFERMERA/O                                    1
      AUXILIAR DE NUTRICION                          1
      TECNICO/A ADMINISTRATIVO I                     1
      AUXILIAR SANITARIO                             1
      SUPERVISOR

In [241]:
#df_2019.head(10)
df_2019.describe()
#df_2019.columns

,PEA,PLIEGO,UE,ESTADO,Quintil,Dist Frontera,VRAEM 2016 (DS 040-2016-PCM),VRAEM 2017 (DS 112-2017-PCM),"EMERGENCIA\n (*1*)D.S. 136-2019-PCM \nDesde el: 26 Julio Hasta el: 24 Set \ny (*2*) D.S. 135-2019-PCM \nDesde el: 28 Julio, Hasta el: 25 Set\n(*3*) D.S. 137-2019-PCM \nDesde el: 27 Julio, Hasta el: 24 Set",ZAF 2014 FINAL,ESTRATEGICOS,MICRORRED PRIORIZADA APS,APS 2015,id_condicion_especialidad,fecha_nacimiento_dt,EDAD
count,216454.0,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,31172.000000,215728,215728.000000
mean,1.0,323.753574,1002.552237,0.998425,3.304882,0.051267,0.028514,0.028630,0.028935,0.055074,0.223521,0.173464,0.143924,1.317914,1975-08-13 21:50:58.251131040,43.913715
min,1.0,11.000000,117.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1933-03-09 00:00:00,17.000000
25%,1.0,11.000000,787.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1966-02-27 00:00:00,34.000000
50%,1.0,446.000000,1006.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1977-01-07 00:00:00,43.000000
75%,1.0,454.000000,1322.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,1985-07-21 00:00:00,53.000000
max,1.0,464.000000,1708.000000,1.000000,5.000000,1.000000,2.000000,2.000000,3.000000,1.000000,1.000000,2.000000,2.000000,4.000000,2002-03-16 00:00:00,86.000000
std,0.0,197.553961,477.622146,0.039660,1.413957,0.220543,0.218606,0.218496,0.251120,0.228125,0.416605,0.475800,0.434761,0.786982,NaN,12.059142


In [242]:
# 2.4 AUDITORÍA DE VALORES (Aquí ya funciona perfectamente porque los DFs tienen datos)
print("\n2. COMPARANDO CONTENIDO REAL DE COLUMNAS SOSPECHOSAS:\n")
df_analisis = comparar_valores(df_2019, 'PLIEGO', '2019', df_2025, 'PLIEGO', '2025')

# Mostrar los primeros registros del cruce de datos para validar
df_analisis.head(20)

# ------- OTRA FORMA --------
#comparar_valores(df_2019, 'id_cargo', '2019', df_2025, 'id_cargo_recod', '2025')



2. COMPARANDO CONTENIDO REAL DE COLUMNAS SOSPECHOSAS:

2019 (PLIEGO): 30 valores únicos
2025 (PLIEGO): 30 valores únicos
¿Son el mismo conjunto? True


,2019 (PLIEGO),2025 (PLIEGO)
0,11,11
1,131,131
2,134,134
3,135,135
4,136,136
5,440,440
6,441,441
7,442,442
8,443,443
9,444,444


In [243]:
# 2.5 Mapeo de columnas de canonicas y de descarte

mapeo_columnas = {
    2019: {

        'CALSIFICACION': 'CLASIFICACION',     # confirmado: CALSIFICACION (2019) = CLASIFICACION (2025)
        'DESCRIPCION ESTABLECIMIENTO': 'DESCRIPCIONESTABLECIMIENTO', # confirmado: DESCRIPCION ESTABLECIMIENTO (2019) = DESCRIPCIONESTABLECIMIENTO (2025)
        'Dist Frontera'    : 'DistFrontera',  # confirmado: Dist Frontera(2019) =  DistFrontera(2025)
        'Grupo Final'      : 'GrupoFinal2', # confirmado: Grupo Final(2019) = GrupoFinal2(2025)
        'Grupo Final 2'    : 'GrupoFinal3',   # confirmado: Grupo Final(2019) = GrupoFinal3(2025)
        'PLIEGO + DESCRIP' : 'PLIEGODESCRIP', # confirmado: PLIEGO + DESCRIP (2019) = PLIEGODESCRIP (2025)
        'REANES FINAL'     : 'RENAES',        # confirmado: REANES FINAL(2019) = RENAES (2025) *** Debe ser de 8 digitos NO NO NO numerico******
        'UE + DESCRIP UE'  : 'UEDESCRIPUE',   # confirmado: UE + DESCRIP UE(2019) = UEDESCRIPUE(2025)
        'ZAF 2014 FINAL'   : 'ZAF2014FINAL',  # confirmado: ZAF 2014 FINAL(2019) = ZAF2014FINAL(2025)  
        'cargo'            : 'CARGO',         # Solo modificamos a mayuscula por estilo
        'id_cargo'         : 'ID_CARGO'       # Solo modificamos a mayuscula por estilo
    },
    2025: {
        # 2025 ya usa los nombres canónicos para todos estos campos, no necesita entradas
        'cargo_recod'     : 'CARGO',    # confirmado: cargo_recod(2025) = cargo(2019)
        'id_cargo_recod'  : 'ID_CARGO', # confirmado: id_cargo_recod(2025) = id_cargo(2019)
        'Edad'            : 'EDAD'      # Se mantiene porque en excel 2019 se obtuvo EDAD,
    
    },
}

In [244]:
columnas_descartadas = {
    2019: {
        'APS 2015'                      : 'Solo existe en 2019, sin equivalente en años posteriores',
        'CARGO_ESTRUCTURAL'             : 'No existe en BD 2025 aunque es importante',
        'CODCARGO'                      : 'No es necesario tiene codigo errados',
        'DESCRIPCION PLIEGO'            : 'No necesario porque es lo mismo que PLIEGO + DESCRIP',

        'EMERGENCIA (*1*)D.S. 136-2019-PCM Desde el: 26 Julio Hasta el: 24 Set y (*2*) D.S. 135-2019-PCM Desde el: 28 Julio, Hasta el: 25 Set (*3*) '
        'D.S. 137-2019-PCM Desde el: 27 Julio, Hasta el: 24 Set': 'Campo no necesario porque cada 3 meses se actualiza y no está actualizado',
        
        'ESTADO'                        : 'No necesario, sin equivalente en años posteriores',
        'INSTITUCION'                   : 'No necesario, sin equivalente en años posteriores',
        'MICRORRED PRIORIZADA APS'      : 'No necesario, sin equivalente en años posteriores',
        'VRAEM 2016 (DS 040-2016-PCM)'  : 'No necesario, sin equivalente en años posteriores',
        'VRAEM 2017 (DS 112-2017-PCM)'  : 'No necesario, sin equivalente en años posteriores',
        'id_condicion_especialidad'     : 'No necesario, sin equivalente en años posteriores',
        'fecha_nacimiento'              : 'Transformado a fecha_nacimiento_dt y luego a EDAD; no se conserva en el consolidado final',
        'UNIDAD EJECUTORA'              : 'Duplicado de UE + DESCRIP UE',
        'UE'                            : 'No necesario, UE + DESCRIP UE ya contiene ese datos',
        'PLIEGO'                        : 'No necesario, PLIEGO + DESCRIP ya contiene ese datos'
    },
    2025: {
        'COMUNIDAD_INDIGENAREFERENCIADGAINPORUBIGEOFEBRERO2022' : 'No necesario, sin equivalente en años anteriores',
        'DESCRIPCIONPLIEGO'                                     : 'Obtenido de PLIEGODESCRIP',
        'DOBLEEMPLEO'                                           : 'No necesario, sin equivalente en años anteriores',
        'EESSCLASJULIO2022'                                     : 'No necesario, sin equivalente en años anteriores',
        'EMERGENCIA1D.S.117123133'                              : 'No necesario, sin equivalente en años anteriores',
        'FRIAJEPORUBIGEO20222024'                               : 'No necesario, sin equivalente en años anteriores',
        'GrupoFinal1'                                           : 'No necesario, se puede obtener de GrupoFinal2',
        'Grupoetareo'                                           : 'No tiene equivalente. Muy necesario, pero se puede obtener de EDAD (contiene grupo por edad)',
        'HELADASPORUBIGEO20222024'                              : 'No necesario, sin equivalente en años anteriores',
        'NIVEL'                                                 : 'No necesario, sin equivalente en años anteriores. Se puede obtener de CATEGORIA',
        'RISAL11NOVIEMBRE2024'                                  : 'No necesario, sin equivalente en años anteriores',
        'VRAEM2022DS1332022PCM'                                 : 'No necesario, sin equivalente en años anteriores',
        'profesion'                                             : 'No necesario, sin equivalente en años anteriores',
        'UNIDADEJECUTORA'                                       : 'Duplicado de UEDESCRIPUE',
        'UE'                                                    : 'No necesario, UEDESCRIPUE ya contiene ese datos',
        'PLIEGO'                                                : 'No necesario, PLIEGODESCRIP ya contiene ese datos'
    },
}

In [245]:
# =============== Descartando columnas con prefijos 'EMERGENCIA' ==============
# =======================================================================
descartar_por_prefijo(2019,'EMERGENCIA', 'Campo no necesario porque cada 3 meses se actualiza y no está actualizado',columnas_2019)
descartar_por_prefijo(2025,'EMERGENCIA', 'Campo no necesario porque cada 3 meses se actualiza y no está actualizado',columnas_2025)


2019: descartada -> 'EMERGENCIA\n (*1*)D.S. 136-2019-PCM \nDesde el: 26 Julio  Hasta el: 24 Set \ny (*2*) D.S. 135-2019-PCM \nDesde el: 28 Julio, Hasta el: 25 Set\n(*3*) D.S. 137-2019-PCM \nDesde el: 27 Julio, Hasta el: 24 Set'
2025: descartada -> 'EMERGENCIA1D.S.117123133Desdeel20SetiembreHastael25Enero2026Algu'


['EMERGENCIA1D.S.117123133Desdeel20SetiembreHastael25Enero2026Algu']

In [246]:
# =============== Verificando columnas que de se mantienen ==============
# =======================================================================

# columnas_que_se_mantienen(columnas_2019, 2019)
columnas_que_se_mantienen(columnas_2025, 2025)

['CATEGORIA',
 'CLASIFICACION',
 'DEPARTAMENTO',
 'DESCRIPCIONESTABLECIMIENTO',
 'DIRESA',
 'DISTRITO',
 'DistFrontera',
 'ESTRATEGICOS',
 'GrupoFinal2',
 'GrupoFinal3',
 'MICRORRED',
 'PCM',
 'PEA',
 'PLIEGODESCRIP',
 'PROVINCIA',
 'Quintil',
 'RED',
 'RENAES',
 'TIPO',
 'UBIGEO',
 'UEDESCRIPUE',
 'ZAF2014FINAL',
 'condicion_especialidad',
 'condicion_laboral',
 'es_especialista',
 'especialidad',
 'id_especialidad',
 'regimen_laboral',
 'sexo']

In [247]:
# ****** Descartamos la columna fecha_nacimiento_dt ********
columnas_descartadas[2019]['fecha_nacimiento_dt'] = 'Columna intermedia, usada solo para calcular EDAD'

In [248]:
finales_2019 = set(obtener_columnas_finales(list(df_2019.columns), 2019, mapeo_columnas, columnas_descartadas))
finales_2025 = set(obtener_columnas_finales(list(df_2025.columns), 2025, mapeo_columnas, columnas_descartadas))

print(f"2019 quedaría con {len(finales_2019)} columnas")
print(f"2025 quedaría con {len(finales_2025)} columnas")
print(f"¿Coinciden exactamente? {finales_2019 == finales_2025}")
print(f"En 2019 pero no en 2025: {finales_2019 - finales_2025}")
print(f"En 2025 pero no en 2019: {finales_2025 - finales_2019}")


2019 quedaría con 32 columnas
2025 quedaría con 32 columnas
¿Coinciden exactamente? True
En 2019 pero no en 2025: set()
En 2025 pero no en 2019: set()


In [249]:
finales_2019

{'CARGO',
 'CATEGORIA',
 'CLASIFICACION',
 'DEPARTAMENTO',
 'DESCRIPCIONESTABLECIMIENTO',
 'DIRESA',
 'DISTRITO',
 'DistFrontera',
 'EDAD',
 'ESTRATEGICOS',
 'GrupoFinal2',
 'GrupoFinal3',
 'ID_CARGO',
 'MICRORRED',
 'PCM',
 'PEA',
 'PLIEGODESCRIP',
 'PROVINCIA',
 'Quintil',
 'RED',
 'RENAES',
 'TIPO',
 'UBIGEO',
 'UEDESCRIPUE',
 'ZAF2014FINAL',
 'condicion_especialidad',
 'condicion_laboral',
 'es_especialista',
 'especialidad',
 'id_especialidad',
 'regimen_laboral',
 'sexo'}

## **<font color='#eda985'> 3. Fase de Homologación Secuencial (Año a Año) </font>**

## **<font color='#96f499'> 3.1. Comparación: 2019 vs 2020 </font>**

In [250]:
# =============================================================================
# 3.1. COMPARACIÓN: 2019 VS 2020
# =============================================================================

# 1. Comparación rápida de nombres

columnas_canonicas = finales_2019  # ya es igual a finales_2025, estándar de referencia para todos los años
columnas_2020 = obtener_columnas_iniciales(archivos[2020][0], archivos[2020][1])

print("1. COMPARANDO ESTRUCTURA DE NOMBRES:\n")
comparar_columnas_anios(columnas_canonicas, columnas_2020, 'CANÓNICO', 2020)


1. COMPARANDO ESTRUCTURA DE NOMBRES:

---- Coinciden exactamente (21) ----
  CATEGORIA
  CLASIFICACION
  DEPARTAMENTO
  DESCRIPCIONESTABLECIMIENTO
  DIRESA
  DISTRITO
  ESTRATEGICOS
  MICRORRED
  PCM
  PEA
  PROVINCIA
  Quintil
  RED
  TIPO
  UBIGEO
  condicion_especialidad
  es_especialista
  especialidad
  id_especialidad
  regimen_laboral
  sexo
---- Solo en CANÓNICO (11) ----
  CARGO
  DistFrontera
  EDAD
  GrupoFinal2
  GrupoFinal3
  ID_CARGO
  PLIEGODESCRIP
  RENAES
  UEDESCRIPUE
  ZAF2014FINAL
  condicion_laboral
---- Solo en 2020 (17) ----
  Dist Frontera
  EMERGENCIA
 (*1*)D.S. 093-2020-PCM 
Desde el: 22 Mayo Hasta el: 20 Julio
y (*2*) D.S. 109-2020-PCM 
Desde el: 22 Junio Hasta el: 20 Agosto
(*3*) D.S. 092-2020-PCM 
Desde el: 22 Mayo Hasta el: 20 Juilo      
(*4*) D.S. 105-2020-PCM 
Desde el: 13 Junio Hasta el: 11 Agosto
  GRUPO ETAREO
  Grupo Final 1
  Grupo Final 2
  Grupo Final 3
  NIVEL
  PLIEGO + DESCRIP
  UE + DESCRIP UE
  ZAF 2014 FINAL
  cargo
  codigo_renaes
  condic

In [251]:
# 2. Cargar DataFrame 2020 para poder analizar los valores internos

df_2020 = cargar_dataframe_limpio(2020, archivos)


In [252]:
df_2020

,codigo_renaes,PLIEGO + DESCRIP,UE + DESCRIP UE,UBIGEO,DEPARTAMENTO,PROVINCIA,DISTRITO,DIRESA,RED,MICRORRED,...,condicion_especialidad,regimen_laboral,condicion_laboral formal,condicion_laboral rep 2020,id_cargo,cargo,PEA,Grupo Final 1,Grupo Final 2,Grupo Final 3
0,00005576,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,0951 HOSPITAL DE APOYO DE PUCALLPA,250101,UCAYALI,CORONEL PORTILLO,CALLERIA,UCAYALI,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,...,NaN,Regimen 276,Nombrado,Nombrado,CA157,AUXILIAR ADMINISTRATIVO,1,Auxiliar Administrativo,Auxiliar Administrativo,Auxiliar Administrativo
1,06340000,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,0950 SALUD UCAYALI,250101,UCAYALI,CORONEL PORTILLO,CALLERIA,UCAYALI,NaN,NaN,...,NaN,Regimen 276,Nombrado,Nombrado,CNN8,TECNICO ADMINISTRATIVO NO ESPECIFICADO,1,Tecnico Administrativo,Tecnico Administrativo,Tecnico Administrativo
2,06340000,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,0950 SALUD UCAYALI,250101,UCAYALI,CORONEL PORTILLO,CALLERIA,UCAYALI,NaN,NaN,...,NaN,Regimen 276,Nombrado,Nombrado,CA058,TECNICO/A SANITARIO AMBIENTAL I,1,Tecnico Asistencial,Tecnico Asistencial,Tecnico Asistencial
3,06340000,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,0950 SALUD UCAYALI,250101,UCAYALI,CORONEL PORTILLO,CALLERIA,UCAYALI,NaN,NaN,...,NaN,Regimen 276,Nombrado,Nombrado,CA155,TECNICO/A ADMINISTRATIVO II,1,Tecnico Administrativo,Tecnico Administrativo,Tecnico Administrativo
4,00005576,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,0951 HOSPITAL DE APOYO DE PUCALLPA,250101,UCAYALI,CORONEL PORTILLO,CALLERIA,UCAYALI,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,...,NaN,Regimen 276,Nombrado,Nombrado,CA196,TECNICO/A EN ENFERMERIA II,1,Tecnico Asistencial,Tecnico Asistencial,Tecnico Asistencial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262861,01000000,011 M. DE SALUD,0117 ADMINISTRACION CENTRAL - MINSA,150113,LIMA,LIMA,JESUS MARIA,MINSA ADMINISTRATIVO,NaN,NaN,...,NaN,Regimen 1057 (CAS),Contrato CAS,Contrato CAS,CA168,ENFERMERA/O,1,Profesional Asistencial,Profesional Asistencial,Enfermero
262862,01000000,011 M. DE SALUD,0117 ADMINISTRACION CENTRAL - MINSA,150113,LIMA,LIMA,JESUS MARIA,MINSA ADMINISTRATIVO,NaN,NaN,...,NaN,Regimen 1057 (CAS),Contrato CAS,Contrato CAS,CA165,MEDICO,1,MEDICO,Profesional Asistencial,Médico
262863,00006207,011 M. DE SALUD,0143 HOSPITAL NACIONAL ARZOBISPO LOAYZA,150101,LIMA,LIMA,LIMA,LIMA CENTRO,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,...,Residente,Regimen 276,Residente,Residente,CA165,MEDICO,1,MEDICO,Profesional Asistencial,Médico
262864,00006210,136 INSTITUTO NACIONAL DE ENFERMEDADES NEOPLAS...,1235 INSTITUTO NACIONAL DE ENFERMEDADES NEOPLA...,150141,LIMA,LIMA,SURQUILLO,LIMA CENTRO,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,...,Residente,Regimen 276,Residente,Residente,CA165,MEDICO,1,MEDICO,Profesional Asistencial,Médico


In [253]:
# Veririfincando carga
df_2020.count()

codigo_renaes                                                                                                                                                                                                                                                                             262866
PLIEGO + DESCRIP                                                                                                                                                                                                                                                                          262866
UE + DESCRIP UE                                                                                                                                                                                                                                                                           262866
UBIGEO                                                                                                                               

In [254]:
# Veririfincando carga
df_2020.describe()

,Quintil,Dist Frontera,EMERGENCIA\n (*1*)D.S. 093-2020-PCM \nDesde el: 22 Mayo Hasta el: 20 Julio\ny (*2*) D.S. 109-2020-PCM \nDesde el: 22 Junio Hasta el: 20 Agosto\n(*3*) D.S. 092-2020-PCM \nDesde el: 22 Mayo Hasta el: 20 Juilo \n(*4*) D.S. 105-2020-PCM \nDesde el: 13 Junio Hasta el: 11 Agosto,ZAF 2014 FINAL,ESTRATEGICOS,PEA
count,262866.000000,262866.000000,262866.000000,262866.000000,262866.000000,262866.0
mean,3.360610,0.050992,0.055971,0.047149,0.221459,1.0
std,1.395471,0.219981,0.415950,0.211959,0.415229,0.0
min,1.000000,0.000000,0.000000,0.000000,0.000000,1.0
25%,2.000000,0.000000,0.000000,0.000000,0.000000,1.0
50%,3.000000,0.000000,0.000000,0.000000,0.000000,1.0
75%,5.000000,0.000000,0.000000,0.000000,0.000000,1.0
max,5.000000,1.000000,4.000000,1.000000,1.000000,1.0


In [255]:
# ---- Verificando tipo datos de fecha de nacimiento excel 2020 ----
df_2020['fecha_nacimiento'].dtype


<StringDtype(storage='python', na_value=nan)>

In [256]:
# ---- Verificando datos de fecha de nacimiento excel 2020 ----
df_2020[['fecha_nacimiento']]


,fecha_nacimiento
0,24/09/1951
1,07/06/1958
2,15/07/1960
3,18/08/1957
4,11/03/1955
...,...
262861,30/09/1964
262862,05/01/1985
262863,No especifica
262864,No especifica


In [257]:
# Convertimos fecha y calculamos EDAD en el DF de 2020
df_2020 = convertir_fecha_nacimiento(df_2020)
df_2020['EDAD'] = calcular_edad(df_2020['fecha_nacimiento_dt'], 2020)

# 3185 registros no se pueden obtener EDAD porque en el campo fecha_nacimiento indican 'No especifica'


Nulos genuinos: 3183
Nulos despues de convertir: 3185
No se pudieron convertir (formato desconocido): 2


In [258]:
# 3. Analizando el contenido de las columnas
df_analisis_2020 = comparar_valores(df_2019, 'regimen_laboral', '2019', df_2020, 'condicion_laboral rep 2020', '2020')
df_analisis_2020

2019 (regimen_laboral): 6 valores únicos
2020 (condicion_laboral rep 2020): 24 valores únicos
¿Son el mismo conjunto? False


,2019 (regimen_laboral),2020 (condicion_laboral rep 2020)
0,Internos de carreras de la salud,CAS COVID Brigadas Cubanos
1,Regimen 1057 (CAS),CAS COVID Brigadas ENA LLAMOSAS
2,Regimen 276,CAS COVID Hospitales Modulares
3,Regimen 728,Con estipendio (pagado)
4,Servicio de terceros / Locación de servicios,Contratado 276 - Plazo fijo
5,regimen 276,Contratado 276 - Plazo indeterminado
6,NaN,Contrato 728 - Plazo Indeterminado
7,NaN,Contrato CAS
8,NaN,Contrato CLAS
9,NaN,Contrato Municipal CAS


In [259]:
# 4. Inicializar los diccionarios para el año 2020

mapeo_columnas[2020] = {
    # 'COLUMNA_2020': 'NOMBRE_CANÓNICO',
    'cargo'                       : 'CARGO',                      # Solo modificamos a mayuscula por estilo
    'Dist Frontera'               : 'DistFrontera',               # confirmado: Dist Frontera(2020) =  DistFrontera(CANONICO)
    'Grupo Final 2'               : 'GrupoFinal2',                # confirmado: Grupo Final(2020) = GrupoFinal2(CANONICO)
    'Grupo Final 3'               : 'GrupoFinal3',                # confirmado: Grupo Final(2020) = GrupoFinal3(CANONICO)
    'id_cargo'                    : 'ID_CARGO',                   # Solo modificamos a mayuscula por estilo
    'PLIEGO + DESCRIP'            : 'PLIEGODESCRIP',              # confirmado: PLIEGO + DESCRIP (2020) = PLIEGODESCRIP (CANONICO)
    'codigo_renaes'               : 'RENAES',                     # confirmado: REANES FINAL(2020) = RENAES (CANONICO)
    'UE + DESCRIP UE'             : 'UEDESCRIPUE',                # confirmado: UE + DESCRIP UE(2020) = UEDESCRIPUE(CANONICO)
    'ZAF 2014 FINAL'              : 'ZAF2014FINAL',               # confirmado: ZAF 2014 FINAL(2020) = ZAF2014FINAL(CANONICO)  
    'condicion_laboral rep 2020'    : 'condicion_laboral',        # confirmado: condicion_laboral formal = condicion_laboral(CANONICO)
    
    
}

columnas_descartadas[2020] = {
    'GRUPO ETAREO'              : 'No tiene equivalente. Muy necesario, pero se puede obtener de EDAD (contiene grupo por edad)',
    'Grupo Final 1'             : 'NO existe en la lista CANONICA',
    'NIVEL'                     : 'NO existe en la lista CANONICA',
    'condicion_laboral formal'  : 'No necesario es es igual y hasta mas detallado',
    'edad final'                : 'Se calculo la EDAD, esto ya es duplicado',
    'fecha_nacimiento'          : 'Transformado a fecha_nacimiento_dt y luego a EDAD; no se conserva en el consolidado final'
    
}

In [260]:
# =============== Descartando columnas con prefijos 'EMERGENCIA' ==============
# =======================================================================
descartar_por_prefijo(2020,'EMERGENCIA', 'Campo no necesario porque cada 3 meses se actualiza y no está actualizado',columnas_2020)

2020: descartada -> 'EMERGENCIA\n (*1*)D.S. 093-2020-PCM \nDesde el: 22 Mayo Hasta el: 20 Julio\ny (*2*) D.S. 109-2020-PCM \nDesde el: 22 Junio Hasta el: 20 Agosto\n(*3*) D.S. 092-2020-PCM \nDesde el: 22 Mayo Hasta el: 20 Juilo      \n(*4*) D.S. 105-2020-PCM \nDesde el: 13 Junio Hasta el: 11 Agosto'


['EMERGENCIA\n (*1*)D.S. 093-2020-PCM \nDesde el: 22 Mayo Hasta el: 20 Julio\ny (*2*) D.S. 109-2020-PCM \nDesde el: 22 Junio Hasta el: 20 Agosto\n(*3*) D.S. 092-2020-PCM \nDesde el: 22 Mayo Hasta el: 20 Juilo      \n(*4*) D.S. 105-2020-PCM \nDesde el: 13 Junio Hasta el: 11 Agosto']

In [261]:
# =============== Verificando columnas que de se mantienen ==============
# =======================================================================

# columnas_que_se_mantienen(columnas_2019, 2019)
columnas_que_se_mantienen(columnas_2020, 2020)

['CATEGORIA',
 'CLASIFICACION',
 'DEPARTAMENTO',
 'DESCRIPCIONESTABLECIMIENTO',
 'DIRESA',
 'DISTRITO',
 'ESTRATEGICOS',
 'MICRORRED',
 'PCM',
 'PEA',
 'PROVINCIA',
 'Quintil',
 'RED',
 'TIPO',
 'UBIGEO',
 'condicion_especialidad',
 'es_especialista',
 'especialidad',
 'id_especialidad',
 'regimen_laboral',
 'sexo']

In [262]:
# ****** Descartamos la columna fecha_nacimiento_dt ********
columnas_descartadas[2020]['fecha_nacimiento_dt'] = 'Columna intermedia, usada solo para calcular EDAD'

In [263]:
finales_2019 = set(obtener_columnas_finales(list(df_2019.columns), 2019, mapeo_columnas, columnas_descartadas))
finales_2025 = set(obtener_columnas_finales(list(df_2025.columns), 2025, mapeo_columnas, columnas_descartadas))
finales_2020 = set(obtener_columnas_finales(list(df_2020.columns), 2020,mapeo_columnas, columnas_descartadas))

print(f"2019 quedaría con {len(finales_2019)} columnas")
print(f"2020 quedaría con {len(finales_2020)} columnas")
print(f"2025 quedaría con {len(finales_2025)} columnas")

print(f"¿Coinciden exactamente? {finales_2019 == finales_2025}")
print(f"En 2019 pero no en 2025: {finales_2019 - finales_2025}")
print(f"En 2025 pero no en 2019: {finales_2025 - finales_2019}")
print('='*50)
print('='*50)

print(f"¿Coinciden exactamente? {finales_2019 == finales_2020 == finales_2025}")
print(f"En 2019 pero no en 2020: {finales_2019 - finales_2020}")
print(f"En 2020 pero no en 2019: {finales_2020 - finales_2019}")


2019 quedaría con 32 columnas
2020 quedaría con 32 columnas
2025 quedaría con 32 columnas
¿Coinciden exactamente? True
En 2019 pero no en 2025: set()
En 2025 pero no en 2019: set()
¿Coinciden exactamente? True
En 2019 pero no en 2020: set()
En 2020 pero no en 2019: set()


## **<font color='#96f499'> 3.2. Comparación: 2020 vs 2021 </font>**

In [264]:
# =============================================================================
# 3.2. COMPARACIÓN: 2020 VS 2021
# =============================================================================

# 1. Comparación rápida de nombres

columnas_canonicas = finales_2020  # ya es igual a finales_2025, estándar de referencia para todos los años
columnas_2021 = obtener_columnas_iniciales(archivos[2021][0], archivos[2021][1])

print("1. COMPARANDO ESTRUCTURA DE NOMBRES:\n")
comparar_columnas_anios(columnas_canonicas, columnas_2021, 'CANÓNICO', 2021)

1. COMPARANDO ESTRUCTURA DE NOMBRES:

---- Coinciden exactamente (25) ----
  CATEGORIA
  CLASIFICACION
  DEPARTAMENTO
  DESCRIPCIONESTABLECIMIENTO
  DIRESA
  DISTRITO
  DistFrontera
  ESTRATEGICOS
  MICRORRED
  PCM
  PEA
  PLIEGODESCRIP
  PROVINCIA
  Quintil
  RED
  TIPO
  UBIGEO
  UEDESCRIPUE
  ZAF2014FINAL
  condicion_especialidad
  es_especialista
  especialidad
  id_especialidad
  regimen_laboral
  sexo
---- Solo en CANÓNICO (7) ----
  CARGO
  EDAD
  GrupoFinal2
  GrupoFinal3
  ID_CARGO
  RENAES
  condicion_laboral
---- Solo en 2021 (11) ----
  GrupoFinal 1
  GrupoFinal 2
  GrupoFinal 3
  NIVEL
  cargo
  codigo_renaes
  condicion_laboralformal
  condicion_laboralrep2020
  edadfinal
  id_cargo
  id_condicion_especialidad


In [265]:
# 2. Cargar DataFrame 2020 para poder analizar los valores internos

df_2021 = cargar_dataframe_limpio(2021, archivos)

In [266]:
df_2021

,codigo_renaes,PLIEGODESCRIP,UEDESCRIPUE,UBIGEO,DEPARTAMENTO,PROVINCIA,DISTRITO,DIRESA,RED,MICRORRED,...,condicion_especialidad,regimen_laboral,condicion_laboralformal,condicion_laboralrep2020,id_cargo,cargo,PEA,GrupoFinal 1,GrupoFinal 2,GrupoFinal 3
0,00005885,011 M. DE SALUD,1686 DIRECCION DE REDES INTEGRADAS DE SALUD LI...,150103,LIMA,LIMA,ATE,LIMA ESTE,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,...,NaN,Servicio de terceros / Locación de servicios,Servicio de terceros / locación de servicios,Servicio de terceros / locación de servicios,CA165,MEDICO,1,MEDICO,Profesional Asistencial,Médico
1,00003301,458 GOBIERNO REGIONAL DEL DEPARTAMENTO DE PUNO,0917 SALUD SAN ROMAN,211101,PUNO,SAN ROMAN,JULIACA,PUNO,SAN ROMAN,SANTA ADRIANA,...,NaN,Regimen 1057 (CAS),Contrato CAS,Contrato CAS,CA170,OBSTETRA,1,Profesional Asistencial,Profesional Asistencial,Obstetra
2,00001282,443 GOBIERNO REGIONAL DEL DEPARTAMENTO DE AREQ...,1222 SALUD RED PERIFERICA AREQUIPA,040117,AREQUIPA,AREQUIPA,SACHACA,AREQUIPA,AREQUIPA CAYLLOMA,YANAHUARA,...,NaN,Regimen 276,Serums Equivalente 2021 - II,Serums Equivalente 2021 - II,CA179,TRABAJADOR/A SOCIAL,1,Profesional Asistencial,Profesional Asistencial,Trabajadora Social
3,00023014,461 GOBIERNO REGIONAL DEL DEPARTAMENTO DE TUMBES,0940 SALUD TUMBES,240101,TUMBES,TUMBES,TUMBES,TUMBES,TUMBES,PAMPA GRANDE,...,NaN,Regimen 1057 (CAS),Contrato CAS,Contrato CAS,CA203,TECNICO/A EN FARMACIA I,1,Tecnico Asistencial,Tecnico Asistencial,Tecnico Asistencial
4,00006215,011 M. DE SALUD,0149 HOSPITAL NACIONAL DOCENTE MADRE NIÑO - SA...,150101,LIMA,LIMA,LIMA,LIMA CENTRO,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,...,NaN,Regimen 276,Nombrado,Nombrado,CA047,EDUCADOR/A PARA LA SALUD I,1,Profesional Administrativo,Profesional Administrativo,Profesional Administrativo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294909,00023159,011 M. DE SALUD,1670 HOSPITAL DE EMERGENCIAS VILLA EL SALVADOR,150142,LIMA,LIMA,VILLA EL SALVADOR,LIMA SUR,NO PERTENECE A NINGUNA RED,NO PERTENECE A NINGUNA MICRORED,...,CONSTANCIA,Regimen 1057 (CAS),Contrato CAS,Contrato CAS,CA166,ENFERMERA/O ESPECIALISTA,1,Profesional Asistencial,Profesional Asistencial,Enfermero
294910,00000442,450 GOBIERNO REGIONAL DEL DEPARTAMENTO DE JUNIN,1613 GOB. REG. DE JUNIN - RED DE SALUD SAN MAR...,120606,JUNIN,SATIPO,PANGOA,JUNIN,SAN MARTIN DE PANGOA,NO PERTENECE A NINGUNA MICRORED,...,RNE,Regimen 276,Nombrado,Nombrado,CA165,MEDICO,1,MEDICO,Profesional Asistencial,Médico
294911,00000308,450 GOBIERNO REGIONAL DEL DEPARTAMENTO DE JUNIN,0828 SALUD CHANCHAMAYO,120301,JUNIN,CHANCHAMAYO,CHANCHAMAYO,JUNIN,CHANCHAMAYO,NO PERTENECE A NINGUNA MICRORED,...,NaN,Regimen 1057 (CAS),Contrato CAS,Contrato CAS,CA165,MEDICO,1,MEDICO,Profesional Asistencial,Médico
294912,00000308,450 GOBIERNO REGIONAL DEL DEPARTAMENTO DE JUNIN,0828 SALUD CHANCHAMAYO,120301,JUNIN,CHANCHAMAYO,CHANCHAMAYO,JUNIN,CHANCHAMAYO,NO PERTENECE A NINGUNA MICRORED,...,NaN,Regimen 276,Nombrado,Nombrado,CA165,MEDICO,1,MEDICO,Profesional Asistencial,Médico


In [267]:
# Verificando carga
df_2021.count()

codigo_renaes                 294914
PLIEGODESCRIP                 294914
UEDESCRIPUE                   294914
UBIGEO                        294914
DEPARTAMENTO                  294914
PROVINCIA                     294914
DISTRITO                      294914
DIRESA                        294914
RED                           256140
MICRORRED                     245759
CLASIFICACION                 294800
TIPO                          245759
DESCRIPCIONESTABLECIMIENTO    294914
CATEGORIA                     294914
NIVEL                         294914
Quintil                       294914
PCM                           294914
DistFrontera                  294914
ZAF2014FINAL                  294914
ESTRATEGICOS                  294914
edadfinal                     294793
sexo                          294914
es_especialista               273238
id_especialidad                36459
especialidad                   36481
id_condicion_especialidad     294914
condicion_especialidad         36634
r

In [268]:
# Verificando carga
df_2021.describe()

,Quintil,DistFrontera,ZAF2014FINAL,ESTRATEGICOS,edadfinal,PEA
count,294914.000000,294914.000000,294914.000000,294914.000000,294793.000000,294914.0
mean,3.390999,0.047075,0.045790,0.220156,41.611646,1.0
std,1.403748,0.211799,0.209029,0.414352,12.154998,0.0
min,1.000000,0.000000,0.000000,0.000000,18.000000,1.0
25%,2.000000,0.000000,0.000000,0.000000,32.000000,1.0
50%,3.000000,0.000000,0.000000,0.000000,39.000000,1.0
75%,5.000000,0.000000,0.000000,0.000000,50.000000,1.0
max,5.000000,1.000000,1.000000,1.000000,88.000000,1.0


In [269]:
# 3. Analizando el contenido de las columnas
df_analisis_2021 = comparar_valores(df_2020, 'UE + DESCRIP UE', '2020', df_2021, 'UEDESCRIPUE', '2021')
df_analisis_2021


2020 (UE + DESCRIP UE): 231 valores únicos
2021 (UEDESCRIPUE): 230 valores únicos
¿Son el mismo conjunto? False


,2020 (UE + DESCRIP UE),2021 (UEDESCRIPUE)
0,0117 ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA
1,0121 INSTITUTO NACIONAL DE SALUD MENTAL,0121 INSTITUTO NACIONAL DE SALUD MENTAL
2,0123 INSTITUTO NACIONAL DE CIENCIAS NEUROLOGICAS,0123 INSTITUTO NACIONAL DE CIENCIAS NEUROLOGICAS
3,0124 INSTITUTO NACIONAL DE OFTALMOLOGIA,0124 INSTITUTO NACIONAL DE OFTALMOLOGIA
4,0125 INSTITUTO NACIONAL DE REHABILITACION,0125 INSTITUTO NACIONAL DE REHABILITACION
...,...,...
226,1696 GOB.REG.DPTO.HUANUCO - RED DE SALUD PACHI...,1696 GOB.REG.DPTO.HUANUCO - RED DE SALUD PACHI...
227,1708 GOB. REG. DEL DPTO. DE MADRE DE DIOS - RE...,1712 GOB. REG. CAJAMARCA - SALUD CAJAMARCA - C...
228,1712 GOB. REG. CAJAMARCA - SALUD CAJAMARCA - C...,1714 RED DE SALUD LORETO - NAUTA
229,1714 RED DE SALUD LORETO - NAUTA,1726 HOSPITAL EMERGENCIA ATE VITARTE


In [270]:
# 4. Inicializar los diccionarios para el año 2021

mapeo_columnas[2021] = {
    # 'COLUMNA_2021': 'NOMBRE_CANÓNICO',

    'cargo'                       : 'CARGO',                      # Solo modificamos a mayuscula por estilo
    'edadfinal'                   : 'EDAD',                       #confirmado: edadfinal(2021) = EDAD(CANONICO)
    'GrupoFinal 2'                : 'GrupoFinal2',                # confirmado: Grupo Final(2021) = GrupoFinal2(CANONICO)
    'GrupoFinal 3'                : 'GrupoFinal3',                # confirmado: Grupo Final(2021) = GrupoFinal3(CANONICO)
    'id_cargo'                    : 'ID_CARGO',                   # Solo modificamos a mayuscula por estilo
    'codigo_renaes'               : 'RENAES',                     # confirmado: codigo_renaes(2021) = RENAES (CANONICO)
    'condicion_laboralrep2020'    : 'condicion_laboral'           # confirmado: condicion_laboralrep2020(2021) = condicion_laboral(CANONICO)
    
}

columnas_descartadas[2021] = {

    'GrupoFinal 1'             : 'NO existe en la lista CANONICA',
    'NIVEL'                     : 'NO existe en la lista CANONICA',
    'condicion_laboralformal'   : 'No necesario es igual condicion_laboralrep2020 y hasta mas detallado',
    'id_condicion_especialidad' : 'NO existe en la lista CANONICA'

}

In [271]:
# =============== Descartando columnas con prefijos 'EMERGENCIA' ==============
# =======================================================================
descartar_por_prefijo(2021,'EMERGENCIA', 'Campo no necesario porque cada 3 meses se actualiza y no está actualizado',columnas_2021)

⚠️ No se encontró ninguna columna con prefijo 'EMERGENCIA' en 2021


[]

In [272]:
# =============== Verificando columnas que de se mantienen ==============
# =======================================================================

# columnas_que_se_mantienen(columnas_2019, 2019)
columnas_que_se_mantienen(columnas_2021, 2021)

['CATEGORIA',
 'CLASIFICACION',
 'DEPARTAMENTO',
 'DESCRIPCIONESTABLECIMIENTO',
 'DIRESA',
 'DISTRITO',
 'DistFrontera',
 'ESTRATEGICOS',
 'MICRORRED',
 'PCM',
 'PEA',
 'PLIEGODESCRIP',
 'PROVINCIA',
 'Quintil',
 'RED',
 'TIPO',
 'UBIGEO',
 'UEDESCRIPUE',
 'ZAF2014FINAL',
 'condicion_especialidad',
 'es_especialista',
 'especialidad',
 'id_especialidad',
 'regimen_laboral',
 'sexo']

In [273]:
finales_2021 = set(obtener_columnas_finales(list(df_2021.columns), 2021,mapeo_columnas, columnas_descartadas))

print(f"2019 quedaría con {len(finales_2019)} columnas")
print(f"2020 quedaría con {len(finales_2020)} columnas")
print(f"2021 quedaría con {len(finales_2021)} columnas")
print(f"2025 quedaría con {len(finales_2025)} columnas")


print('='*50)
print('='*50)

print(f"¿Coinciden exactamente? {finales_2020 == finales_2021}")
print(f"En 2020 pero no en 2021: {finales_2020 - finales_2021}")
print(f"En 2021 pero no en 2020: {finales_2021 - finales_2020}")

print('='*50)
print('='*50)
print(f"¿Coinciden exactamente 2019,2020,2021,2025? {finales_2019 == finales_2020 == finales_2021 == finales_2025}")


2019 quedaría con 32 columnas
2020 quedaría con 32 columnas
2021 quedaría con 32 columnas
2025 quedaría con 32 columnas
¿Coinciden exactamente? True
En 2020 pero no en 2021: set()
En 2021 pero no en 2020: set()
¿Coinciden exactamente 2019,2020,2021,2025? True


## **<font color='#96f499'> 3.3. Comparación: 2021 vs 2022 </font>**

In [274]:
# =============================================================================
# 3.1. COMPARACIÓN: 2021 VS 2022
# =============================================================================

# 1. Comparación rápida de nombres

columnas_canonicas = finales_2021  # ya es igual a finales_2025, estándar de referencia para todos los años
columnas_2022 = obtener_columnas_iniciales(archivos[2022][0], archivos[2022][1])

print("1. COMPARANDO ESTRUCTURA DE NOMBRES:\n")
comparar_columnas_anios(columnas_canonicas, columnas_2022, 'CANÓNICO', 2022)

1. COMPARANDO ESTRUCTURA DE NOMBRES:

---- Coinciden exactamente (28) ----
  CATEGORIA
  CLASIFICACION
  DEPARTAMENTO
  DESCRIPCIONESTABLECIMIENTO
  DIRESA
  DISTRITO
  DistFrontera
  ESTRATEGICOS
  GrupoFinal2
  GrupoFinal3
  MICRORRED
  PCM
  PEA
  PLIEGODESCRIP
  PROVINCIA
  Quintil
  RED
  TIPO
  UBIGEO
  UEDESCRIPUE
  ZAF2014FINAL
  condicion_especialidad
  condicion_laboral
  es_especialista
  especialidad
  id_especialidad
  regimen_laboral
  sexo
---- Solo en CANÓNICO (4) ----
  CARGO
  EDAD
  ID_CARGO
  RENAES
---- Solo en 2022 (12) ----
  CATEGORIA 2
  DESCRIPCIONPLIEGO
  EMERGENCIA1D.S.1252022PCMDesdeel17OctubreHasta16Diciemby2D.S.120
  GrupoFinal1
  NIVEL
  PLIEGO
  UE
  UNIDADEJECUTORA
  cargo
  codigo_renaes
  fecha_nacimiento
  id_cargo


In [275]:
# 2. Cargar DataFrame 2020 para poder analizar los valores internos

df_2022 = cargar_dataframe_limpio(2022, archivos)


In [276]:
df_2022

,codigo_renaes,PLIEGO,DESCRIPCIONPLIEGO,PLIEGODESCRIP,UE,UNIDADEJECUTORA,UEDESCRIPUE,UBIGEO,DEPARTAMENTO,PROVINCIA,...,id_cargo,cargo,PEA,GrupoFinal1,GrupoFinal2,GrupoFinal3,es_especialista,id_especialidad,especialidad,condicion_especialidad
0,06340000,462,GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,950,SALUD UCAYALI,0950 SALUD UCAYALI,250101,UCAYALI,CORONEL PORTILLO,...,CA156,TECNICO/A ADMINISTRATIVO I,1,Tecnico Administrativo,Tecnico Administrativo,Tecnico Administrativo,No,NaN,NaN,NaN
1,06340000,462,GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,950,SALUD UCAYALI,0950 SALUD UCAYALI,250101,UCAYALI,CORONEL PORTILLO,...,CA058,TECNICO/A SANITARIO AMBIENTAL I,1,Tecnico Asistencial,Tecnico Asistencial,Tecnico Asistencial,No,NaN,NaN,NaN
2,06340000,462,GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,950,SALUD UCAYALI,0950 SALUD UCAYALI,250101,UCAYALI,CORONEL PORTILLO,...,CA155,TECNICO/A ADMINISTRATIVO II,1,Tecnico Administrativo,Tecnico Administrativo,Tecnico Administrativo,No,NaN,NaN,NaN
3,00005576,462,GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,951,HOSPITAL DE APOYO DE PUCALLPA,0951 HOSPITAL DE APOYO DE PUCALLPA,250101,UCAYALI,CORONEL PORTILLO,...,CA196,TECNICO/A EN ENFERMERIA II,1,Tecnico Asistencial,Tecnico Asistencial,Tecnico Asistencial,No,NaN,NaN,NaN
4,00005563,462,GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,462 GOBIERNO REGIONAL DEL DEPARTAMENTO DE UCAYALI,1660,GOB. REG. DE UCAYALI - RED DE SALUD N° 01 CORO...,1660 GOB. REG. DE UCAYALI - RED DE SALUD N° 01...,250107,UCAYALI,CORONEL PORTILLO,...,CA168,ENFERMERA/O,1,Profesional Asistencial,Profesional Asistencial,Enfermero,No,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277095,00005617,11,M. DE SALUD,011 M. DE SALUD,1216,HOSPITAL SAN JUAN DE LURIGANCHO,1216 HOSPITAL SAN JUAN DE LURIGANCHO,150132,LIMA,LIMA,...,CA157,AUXILIAR ADMINISTRATIVO,1,Auxiliar Administrativo,Auxiliar Administrativo,Auxiliar Administrativo,NO,NaN,NaN,NaN
277096,00005617,11,M. DE SALUD,011 M. DE SALUD,1216,HOSPITAL SAN JUAN DE LURIGANCHO,1216 HOSPITAL SAN JUAN DE LURIGANCHO,150132,LIMA,LIMA,...,CA157,AUXILIAR ADMINISTRATIVO,1,Auxiliar Administrativo,Auxiliar Administrativo,Auxiliar Administrativo,NO,NaN,NaN,NaN
277097,00005617,11,M. DE SALUD,011 M. DE SALUD,1216,HOSPITAL SAN JUAN DE LURIGANCHO,1216 HOSPITAL SAN JUAN DE LURIGANCHO,150132,LIMA,LIMA,...,CA157,AUXILIAR ADMINISTRATIVO,1,Auxiliar Administrativo,Auxiliar Administrativo,Auxiliar Administrativo,NO,NaN,NaN,NaN
277098,00005617,11,M. DE SALUD,011 M. DE SALUD,1216,HOSPITAL SAN JUAN DE LURIGANCHO,1216 HOSPITAL SAN JUAN DE LURIGANCHO,150132,LIMA,LIMA,...,CA157,AUXILIAR ADMINISTRATIVO,1,Auxiliar Administrativo,Auxiliar Administrativo,Auxiliar Administrativo,NO,NaN,NaN,NaN


In [277]:
# Veririfincando carga
df_2022.count()

codigo_renaes                                                       277100
PLIEGO                                                              277100
DESCRIPCIONPLIEGO                                                   277100
PLIEGODESCRIP                                                       277100
UE                                                                  277100
UNIDADEJECUTORA                                                     277100
UEDESCRIPUE                                                         277100
UBIGEO                                                              277100
DEPARTAMENTO                                                        277100
PROVINCIA                                                           277100
DISTRITO                                                            277100
DIRESA                                                              277100
RED                                                                 251139
MICRORRED                

In [278]:
# Veririfincando carga
df_2022.describe()

,PLIEGO,UE,Quintil,DistFrontera,EMERGENCIA1D.S.1252022PCMDesdeel17OctubreHasta16Diciemby2D.S.120,ZAF2014FINAL,ESTRATEGICOS,PEA
count,277100.000000,277100.000000,277100.000000,277100.000000,277100.000000,277100.000000,277100.000000,277100.0
mean,327.825045,1005.581270,3.290281,0.047882,0.030014,0.049260,0.231685,1.0
std,195.725646,480.690833,1.407004,0.213516,0.254000,0.216411,0.421910,0.0
min,11.000000,117.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.0
25%,11.000000,798.000000,2.000000,0.000000,0.000000,0.000000,0.000000,1.0
50%,446.000000,1006.000000,3.000000,0.000000,0.000000,0.000000,0.000000,1.0
75%,455.000000,1347.000000,5.000000,0.000000,0.000000,0.000000,0.000000,1.0
max,464.000000,1738.000000,5.000000,1.000000,3.000000,1.000000,1.000000,1.0


In [279]:
# ---- Verificando tipo datos de fecha de nacimiento excel 2022 ----
df_2022['fecha_nacimiento'].dtype

dtype('O')

In [280]:
# ---- Verificando datos de fecha de nacimiento excel 2022 ----
df_2022[['fecha_nacimiento']]

,fecha_nacimiento
0,07/06/1958
1,15/07/1960
2,18/08/1957
3,11/03/1955
4,14/01/1963
...,...
277095,26/08/1991
277096,29/06/1999
277097,28/11/2000
277098,09/12/1994


In [281]:
# Convertimos fecha y calculamos EDAD en el DF de 2022
df_2022 = convertir_fecha_nacimiento(df_2022)
df_2022['EDAD'] = calcular_edad(df_2022['fecha_nacimiento_dt'], 2022)

# Solo un registro no se puede calcular EDAD (30/05/85) deberi ser 30/05/1985

# Con las siguiente lineas puedes verificar cual es el registro con diferente formato
# serie_temp = df_2022['fecha_nacimiento'].replace(r'(?i)no especifica|n/a|sin dato|no aplica', pd.NA, regex=True)
# problematico = serie_temp[serie_temp.notna() & df_2022['fecha_nacimiento_dt'].isna()]
# print(problematico)


Nulos genuinos: 0
Nulos despues de convertir: 1
No se pudieron convertir (formato desconocido): 1


In [282]:
# 3. Analizando el contenido de las columnas
df_analisis_2022 = comparar_valores(df_2021, 'UEDESCRIPUE', '2021', df_2022, 'UNIDADEJECUTORA', '2022')
df_analisis_2022

2021 (UEDESCRIPUE): 230 valores únicos
2022 (UNIDADEJECUTORA): 233 valores únicos
¿Son el mismo conjunto? False


,2021 (UEDESCRIPUE),2022 (UNIDADEJECUTORA)
0,0117 ADMINISTRACION CENTRAL - MINSA,ADMINISTRACION CENTRAL - MINSA
1,0121 INSTITUTO NACIONAL DE SALUD MENTAL,DIRECCION DE RED DE SALUD Nº 03 ATALAYA
2,0123 INSTITUTO NACIONAL DE CIENCIAS NEUROLOGICAS,DIRECCION DE RED DE SALUD Nº 04 AGUAYTIA - SAN...
3,0124 INSTITUTO NACIONAL DE OFTALMOLOGIA,DIRECCION DE REDES INTEGRADAS DE SALUD LIMA CE...
4,0125 INSTITUTO NACIONAL DE REHABILITACION,DIRECCION DE REDES INTEGRADAS DE SALUD LIMA ESTE
...,...,...
228,1714 RED DE SALUD LORETO - NAUTA,SEGURO INTEGRAL DE SALUD
229,1726 HOSPITAL EMERGENCIA ATE VITARTE,SERVICIOS BASICOS DE SALUD CAÑETE-YAUYOS
230,NaN,SERVICIOS BASICOS DE SALUD CHILCA - MALA
231,NaN,SUPERINTENDENCIA NACIONAL DE SALUD


In [283]:
# 4. Inicializar los diccionarios para el año 2022

mapeo_columnas[2022] = {
    # 'COLUMNA_2022': 'NOMBRE_CANÓNICO',
    
    'cargo'                       : 'CARGO',                      # Solo modificamos a mayuscula por estilo
    'id_cargo'                    : 'ID_CARGO',                   # Solo modificamos a mayuscula por estilo
    'codigo_renaes'               : 'RENAES'                      # confirmado: REANES FINAL(2022) = RENAES (CANONICO)
    
}

columnas_descartadas[2022] = {

    'CATEGORIA 2'               : 'No necesario, es duplicado de CATEGORIA',
    'DESCRIPCIONPLIEGO'         : 'No necesario, PLIEGODESCRIP ya contiene la misma informacion',
    'GrupoFinal1'               : 'NO existe en la lista CANONICA',
    'NIVEL'                     : 'NO existe en la lista CANONICA',
    'PLIEGO'                    : 'No necesario, ya contiene la misma informacion en PLIEGODESCRIP',
    'UE'                        : 'No necesario, ya contiene la misma informacion en UEDESCRIPUE',
    'UNIDADEJECUTORA'           : 'No necesario, ya contiene la misma informacion en UEDESCRIPUE',
    'fecha_nacimiento'          : 'Transformado a fecha_nacimiento_dt y luego a EDAD; no se conserva en el consolidado final'
        
}

In [284]:
# =============== Descartando columnas con prefijos 'EMERGENCIA' ==============
# =======================================================================
descartar_por_prefijo(2022,'EMERGENCIA', 'Campo no necesario porque cada 3 meses se actualiza y no está actualizado',columnas_2022)


2022: descartada -> 'EMERGENCIA1D.S.1252022PCMDesdeel17OctubreHasta16Diciemby2D.S.120'


['EMERGENCIA1D.S.1252022PCMDesdeel17OctubreHasta16Diciemby2D.S.120']

In [285]:
# =============== Verificando columnas que de se mantienen ==============
# =======================================================================

# columnas_que_se_mantienen(columnas_2019, 2019)
columnas_que_se_mantienen(columnas_2022, 2022)

['CATEGORIA',
 'CLASIFICACION',
 'DEPARTAMENTO',
 'DESCRIPCIONESTABLECIMIENTO',
 'DIRESA',
 'DISTRITO',
 'DistFrontera',
 'ESTRATEGICOS',
 'GrupoFinal2',
 'GrupoFinal3',
 'MICRORRED',
 'PCM',
 'PEA',
 'PLIEGODESCRIP',
 'PROVINCIA',
 'Quintil',
 'RED',
 'TIPO',
 'UBIGEO',
 'UEDESCRIPUE',
 'ZAF2014FINAL',
 'condicion_especialidad',
 'condicion_laboral',
 'es_especialista',
 'especialidad',
 'id_especialidad',
 'regimen_laboral',
 'sexo']

In [286]:
# ****** Descartamos la columna fecha_nacimiento_dt ********
columnas_descartadas[2022]['fecha_nacimiento_dt'] = 'Columna intermedia, usada solo para calcular EDAD'

In [ ]:

finales_2022 = set(obtener_columnas_finales(list(df_2022.columns), 2022,mapeo_columnas, columnas_descartadas))

print(f"2019 quedaría con {len(finales_2019)} columnas")
print(f"2020 quedaría con {len(finales_2020)} columnas")
print(f"2021 quedaría con {len(finales_2021)} columnas")
print(f"2022 quedaría con {len(finales_2022)} columnas")
print(f"2025 quedaría con {len(finales_2025)} columnas")


print('='*50)
print('='*50)

print(f"¿Coinciden exactamente? {finales_2021 == finales_2022}")
print(f"En 2021 pero no en 2022: {finales_2021 - finales_2022}")
print(f"En 2022 pero no en 2021: {finales_2022 - finales_2021}")

print('='*50)
print('='*50)
print(f"¿Coinciden exactamente 2019,2020,2021,2022,2025? {finales_2019 == finales_2020 == finales_2021 == finales_2022 == finales_2025}")

2019 quedaría con 32 columnas
2020 quedaría con 32 columnas
2021 quedaría con 32 columnas
2022 quedaría con 32 columnas
2025 quedaría con 32 columnas
¿Coinciden exactamente? True
En 2021 pero no en 2022: set()
En 2022 pero no en 2021: set()
¿Coinciden exactamente 2019,2020,2021,2022,2025? True


## **<font color='#96f499'> 3.4. Comparación: 2022 vs 2023 </font>**

## **<font color='#96f499'> 3.5. Comparación: 2023 vs 2024 </font>**

## **<font color='#96f499'> 3.5. Comparación: 2024 vs 2025 </font>**